In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/workspace/default/flight_data/retail_data/all/*.csv")\
    .coalesce(5)
df.createOrReplaceTempView("dfTable")

In [0]:
df.show(5, False)

In [0]:
df.count() == 541909

In [0]:
from pyspark.sql.functions import count
df.select(count("StockCode")).show()

In [0]:
from pyspark.sql.functions import first, last
df.select(first("StockCode"), last("StockCode")).show()

In [0]:
from pyspark.sql.functions import sum_distinct
df.select(sum_distinct("Quantity")).show()

In [0]:
from pyspark.sql.functions import var_pop, var_samp, stddev_pop, stddev_samp
df.select(var_pop("Quantity"), var_samp("Quantity"), stddev_pop("Quantity"), stddev_samp("Quantity")).show()

In [0]:
from pyspark.sql.functions import count, expr
df.groupBy("InvoiceNo").agg(\
    count("Quantity").alias("quan"),
    expr("count(Quantity)")).show(4)

In [0]:
df.select("InvoiceDate").show(5)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc 

windowSpec = Window \
    .partitionBy("CustomerId", "InvoiceDate") \
    .orderBy(desc("Quantity")) \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

In [0]:
from pyspark.sql.functions import col, max
maxPurchaseQuantity = max(col("Quantity")).over(windowSpec)

df.withColumn(
    "maxPurchaseQuantity",
    maxPurchaseQuantity
).show(5)

In [0]:
from pyspark.sql.functions import dense_rank, rank

purchaseDenseRank = dense_rank().over(windowSpec)
purchaseRank = rank().over(windowSpec)

In [0]:
df.where("CustomerId IS NOT NULL").orderBy("CustomerID")\
    .select(
        col("CustomerID"),
        col("InvoiceDate"),
        col("Quantity"),
        purchaseRank.alias("quantityRank"),
        purchaseDenseRank.alias("quantityDenseRank"),
        maxPurchaseQuantity.alias("maxPurchaseQuantity")
    ).show(40)

In [0]:
dfNoNull = df.drop()

In [0]:
dfNoNull.createOrReplaceTempView("dfNoNull")

In [0]:
%sql
SELECT CustomerId, stockCode, sum(Quantity) From dfNoNull
GROUP BY CustomerId, stockCode
ORDER BY CustomerId DESC, stockCode DESC
LIMIT 5

In [0]:
%sql
SELECT CustomerId, stockCode, sum(Quantity) From dfNoNull
GROUP BY GROUPING SETS((CustomerId, stockCode), ())
ORDER BY CustomerId DESC, stockCode DESC

In [0]:
from pyspark.sql.functions import sum
rolledUpDF = dfNoNull.rollup("InvoiceDate", "Country").agg(sum("Quantity"))\
    .selectExpr("InvoiceDate", "Country", "`sum(Quantity)` as total_quantity")\
    .orderBy("InvoiceDate")
rolledUpDF.show(10)

In [0]:
rolledUpDF.where("Country IS NULL").where("InvoiceDate IS NULL").show()

In [0]:
from pyspark.sql.functions import sum
df_clean = df.select("InvoiceDate", "Country", "Quantity")

pivoted = df_clean.groupBy("InvoiceDate").pivot("Country").agg(sum("Quantity"))
pivoted.show()